# VERONICA — Phase 11E.4
## Genuine custom openWakeWord training

This notebook is a cleaned, executable Colab notebook for training `veronica_wake_model`.

It removes the malformed/duplicated code from the supplied file and keeps the real openWakeWord training pipeline.

**Run cells from top to bottom. Do not skip cells.**

In [ ]:
# 1. Create a clean Python 3.11 training environment inside Colab
# This avoids Colab's changing system Python version affecting the training stack.
import os, sys, subprocess, site, platform

VENV = "/content/veronica-venv"
VENV_PY = os.path.join(VENV, "bin", "python")

if not os.path.exists(VENV_PY):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "uv"])
    subprocess.check_call(["uv", "venv", "--python", "3.11", VENV])

print("System Python:", sys.version.split()[0])
print("Training Python:", subprocess.check_output(
    [VENV_PY, "-c", "import sys; print(sys.version.split()[0])"], text=True
).strip())
print("OS:", platform.platform())

subprocess.check_call([VENV_PY, "-m", "pip", "install", "-q", "--upgrade", "pip"])

# Versions chosen to match the flat-layout Piper generator used by the
# first pipeline. Installing them inside the isolated environment keeps
# Colab's system packages out of the training stack.
packages = [
    "numpy<2",
    "scipy",
    "pyyaml",
    "pronouncing",
    "audiomentations==0.33.0",
    "webrtcvad",
    "torch<2",
    "torchaudio<2",
    "piper-phonemize==1.1.0",
    "torchinfo",
    "torchmetrics",
    "onnx",
    "onnxruntime",
    "tqdm",
    "requests",
]

subprocess.check_call([VENV_PY, "-m", "pip", "install", "-q", *packages])

# Make the isolated environment importable by this notebook's Python process.
venv_site = subprocess.check_output(
    [VENV_PY, "-c", "import site; print(site.getsitepackages()[0])"],
    text=True
).strip()
if venv_site not in sys.path:
    sys.path.insert(0, venv_site)

print("PASS: isolated training environment ready")


In [ ]:
# 2. Prepare the exact source trees used by the first pipeline
import os, shutil, subprocess, sys

os.chdir("/content")

PSG_DIR = "/content/piper-sample-generator"
OWW_DIR = "/content/openwakeword"

# Clean previous incomplete clones.
shutil.rmtree(PSG_DIR, ignore_errors=True)
shutil.rmtree(OWW_DIR, ignore_errors=True)

subprocess.check_call([
    "git", "clone", "-q",
    "https://github.com/rhasspy/piper-sample-generator",
    PSG_DIR
])
subprocess.check_call(["git", "-C", PSG_DIR, "fetch", "-q", "--all"])

# The first-pipeline revision was written with a trailing ^.
# Verify both the revision and its parent and select the one that actually
# contains the required flat-layout generate_samples.py.
base = "1a8c49bd29b3a132721086ee88f2253f788594a"
candidates = [base + "^", base]

selected = None
for ref in candidates:
    result = subprocess.run(
        ["git", "-C", PSG_DIR, "ls-tree", "-r", "--name-only", ref, "generate_samples.py"],
        text=True, capture_output=True
    )
    if result.returncode == 0 and result.stdout.strip() == "generate_samples.py":
        selected = ref
        break

if selected is None:
    raise RuntimeError(
        "Neither the requested Piper revision nor its parent contains "
        "generate_samples.py. The notebook will not guess a different source."
    )

subprocess.check_call(["git", "-C", PSG_DIR, "checkout", "-q", selected])
print("PASS: Piper revision:", subprocess.check_output(
    ["git", "-C", PSG_DIR, "rev-parse", "HEAD"], text=True
).strip())
print("PASS: generate_samples.py:", os.path.join(PSG_DIR, "generate_samples.py"))

# Download the LibriTTS generator model used by the first pipeline.
MODEL_DIR = os.path.join(PSG_DIR, "models")
os.makedirs(MODEL_DIR, exist_ok=True)
PIPER_MODEL = os.path.join(MODEL_DIR, "en_US-libritts_r-medium.pt")
PIPER_MODEL_URL = (
    "https://github.com/rhasspy/piper-sample-generator/releases/download/"
    "v2.0.0/en_US-libritts_r-medium.pt"
)

if not os.path.exists(PIPER_MODEL) or os.path.getsize(PIPER_MODEL) < 100_000_000:
    subprocess.check_call([
        "wget", "-q", "--tries=5", "--timeout=300",
        "-O", PIPER_MODEL, PIPER_MODEL_URL
    ])

if os.path.getsize(PIPER_MODEL) < 100_000_000:
    raise RuntimeError("Piper LibriTTS model download is incomplete.")

print(f"PASS: Piper model: {os.path.getsize(PIPER_MODEL)/1e6:.1f} MB")

# Clone official openWakeWord.
subprocess.check_call([
    "git", "clone", "-q",
    "https://github.com/dscripka/openWakeWord",
    OWW_DIR
])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", OWW_DIR])

train_py = os.path.join(OWW_DIR, "openwakeword", "train.py")
if not os.path.isfile(train_py):
    raise RuntimeError("Official openWakeWord train.py was not found.")

print("PASS: openWakeWord:", train_py)


In [ ]:
# 3. Install and verify the official openWakeWord source in the isolated environment
import sys, os, importlib, subprocess

subprocess.check_call([
    "/content/veronica-venv/bin/python",
    "-m", "pip", "install", "-q", "-e", "/content/openwakeword", "--no-deps"
])

sys.path.insert(0, "/content/openwakeword")
importlib.invalidate_caches()

import openwakeword
import onnxruntime
import onnx
import torch

print("PASS: openwakeword:", openwakeword.__file__)
print("PASS: torch:", torch.__version__)
print("PASS: onnxruntime:", onnxruntime.__version__)
print("PASS: onnx:", onnx.__version__)


In [ ]:
# 4. Create a configuration that matches the actual train.py paths
import yaml, os

CONFIG_PATH = "/content/my_model.yaml"
OUTPUT_DIR = "/content/veronica_training"

config = {
    "model_name": "veronica_wake_model",
    "target_phrase": ["Veronica"],
    "custom_negative_phrases": [
        "veronica is not here",
        "verona",
        "veronique",
        "veroni",
        "vero niko",
    ],
    "n_samples": 2000,
    "n_samples_val": 200,
    "tts_batch_size": 16,
    "augmentation_batch_size": 64,
    "piper_sample_generator_path": "/content/piper-sample-generator",
    "output_dir": OUTPUT_DIR,
    "rir_paths": ["/content/openwakeword/openwakeword/resources/rir"],
    "background_paths": ["/content/openwakeword/openwakeword/resources/background_noise"],
    "background_paths_duplication_rate": [1],
    "augmentation_rounds": 10,
    "feature_data_files": {},
    "batch_n_per_class": 100,
    "model_type": "dnn",
    "layer_size": 128,
    "steps": 50000,
    "max_negative_weight": 1000,
    "target_false_positives_per_hour": 0.2,
    # train.py reads this key directly.
    "false_positive_validation_data_path":
        os.path.join(OUTPUT_DIR, "veronica_wake_model", "false_positive_val.npy"),
}

with open(CONFIG_PATH, "w") as f:
    yaml.safe_dump(config, f, sort_keys=False)

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("PASS: configuration:", CONFIG_PATH)
print(open(CONFIG_PATH).read())


In [ ]:
# 5. Verify the resources required before expensive generation
import os, glob, yaml

with open("/content/my_model.yaml") as f:
    cfg = yaml.safe_load(f)

checks = {
    "train.py": "/content/openwakeword/openwakeword/train.py",
    "generate_samples.py": "/content/piper-sample-generator/generate_samples.py",
    "Piper model": "/content/piper-sample-generator/models/en_US-libritts_r-medium.pt",
    "RIR directory": "/content/openwakeword/openwakeword/resources/rir",
    "background directory": "/content/openwakeword/openwakeword/resources/background_noise",
}

for label, path in checks.items():
    if not os.path.exists(path):
        raise RuntimeError(f"Missing {label}: {path}")
    print("PASS:", label)

rir_count = len(glob.glob("/content/openwakeword/openwakeword/resources/rir/*"))
bg_count = len(glob.glob("/content/openwakeword/openwakeword/resources/background_noise/*"))
print("RIR files:", rir_count)
print("Background files:", bg_count)

if rir_count == 0:
    raise RuntimeError("The RIR directory is empty.")
if bg_count == 0:
    raise RuntimeError("The background-noise directory is empty.")


In [ ]:
# 6. Generate the real synthetic training and test clips
import subprocess, sys, os

cmd = [
    "/content/veronica-venv/bin/python",
    "/content/openwakeword/openwakeword/train.py",
    "--training_config", "/content/my_model.yaml",
    "--generate_clips",
]

print("Starting genuine openWakeWord clip generation...")
result = subprocess.run(cmd, text=True)
if result.returncode != 0:
    raise RuntimeError("Clip generation failed. The full error is shown above.")

base = "/content/veronica_training/veronica_wake_model"
clip_dirs = {
    "positive_train": os.path.join(base, "positive_train"),
    "positive_test": os.path.join(base, "positive_test"),
    "negative_train": os.path.join(base, "negative_train"),
    "negative_test": os.path.join(base, "negative_test"),
}

for name, path in clip_dirs.items():
    count = len([x for x in os.listdir(path) if x.lower().endswith(".wav")]) if os.path.isdir(path) else 0
    print(f"{name}: {count} WAV files")
    if count == 0:
        raise RuntimeError(f"No WAV files were generated for {name}.")

print("PASS: real synthetic clip generation")


In [ ]:
# 7. Real augmentation, feature extraction, and false-positive validation data
import subprocess, os, numpy as np

cmd = [
    "/content/veronica-venv/bin/python",
    "/content/openwakeword/openwakeword/train.py",
    "--training_config", "/content/my_model.yaml",
    "--augment_clips",
    "--overwrite",
]

print("Starting genuine feature extraction...")
result = subprocess.run(cmd, text=True)
if result.returncode != 0:
    raise RuntimeError("Augmentation/feature extraction failed. The full error is shown above.")

feature_dir = "/content/veronica_training/veronica_wake_model"
feature_files = [
    "positive_features_train.npy",
    "negative_features_train.npy",
    "positive_features_test.npy",
    "negative_features_test.npy",
]

for name in feature_files:
    path = os.path.join(feature_dir, name)
    if not os.path.isfile(path):
        raise RuntimeError(f"Missing generated feature file: {path}")
    arr = np.load(path, mmap_mode="r")
    print(f"PASS: {name} shape={arr.shape} size={os.path.getsize(path)/1e6:.2f} MB")

# train.py expects a real feature array at false_positive_validation_data_path.
# Build it from the official openWakeWord background-noise clips.
from scipy.io import wavfile
from openwakeword.utils import AudioFeatures

background_dir = "/content/openwakeword/openwakeword/resources/background_noise"
background_wavs = [
    os.path.join(background_dir, f)
    for f in os.listdir(background_dir)
    if f.lower().endswith(".wav")
]

if not background_wavs:
    raise RuntimeError("No official background WAV files were found.")

audio_batch = []
target_len = 32000

for path in background_wavs[:128]:
    sr, data = wavfile.read(path)
    data = np.asarray(data)

    if data.ndim > 1:
        data = data.mean(axis=1)

    # Keep only 16 kHz clips for the validation set.
    if sr != 16000:
        continue

    data = data.astype(np.int16)

    if len(data) < target_len:
        data = np.pad(data, (0, target_len - len(data)))
    else:
        data = data[:target_len]

    audio_batch.append(data)

if not audio_batch:
    raise RuntimeError("No usable 16 kHz background WAV files were found.")

audio_batch = np.stack(audio_batch)
F = AudioFeatures(device="cpu", inference_framework="onnx", ncpu=2)
background_features = F.embed_clips(audio_batch, batch_size=8, ncpu=2)

# Concatenate the background feature frames into the format expected by train.py.
false_positive_path = os.path.join(feature_dir, "false_positive_val.npy")
false_positive_features = background_features.reshape(-1, background_features.shape[-1]).astype(np.float32)
np.save(false_positive_path, false_positive_features)

if not os.path.isfile(false_positive_path) or os.path.getsize(false_positive_path) == 0:
    raise RuntimeError("False-positive validation data was not created.")

print(
    "PASS: false-positive validation data:",
    false_positive_features.shape,
    f"{os.path.getsize(false_positive_path)/1e6:.2f} MB"
)


In [ ]:
# 8. Real model training and ONNX export
import subprocess, sys, os

cmd = [
    "/content/veronica-venv/bin/python",
    "/content/openwakeword/openwakeword/train.py",
    "--training_config", "/content/my_model.yaml",
    "--train_model",
]

print("Starting genuine Veronica model training...")
result = subprocess.run(cmd, text=True)
if result.returncode != 0:
    raise RuntimeError("Model training failed. The full error is shown above.")

MODEL_PATH = "/content/veronica_training/veronica_wake_model.onnx"

if not os.path.isfile(MODEL_PATH) or os.path.getsize(MODEL_PATH) == 0:
    raise RuntimeError("Training finished without a non-empty ONNX model.")

print(f"PASS: trained model: {MODEL_PATH}")
print(f"Size: {os.path.getsize(MODEL_PATH)/1e6:.2f} MB")


In [ ]:
# 9. Genuine ONNX validation
import onnx
import onnxruntime as ort
import numpy as np
import os

MODEL_PATH = "/content/veronica_training/veronica_wake_model.onnx"

model = onnx.load(MODEL_PATH)
onnx.checker.check_model(model)

session = ort.InferenceSession(MODEL_PATH, providers=["CPUExecutionProvider"])

print("PASS: ONNX checker")
print("PASS: ONNX Runtime session")
print("Inputs:")
for x in session.get_inputs():
    print(" ", x.name, x.shape, x.type)
print("Outputs:")
for x in session.get_outputs():
    print(" ", x.name, x.shape, x.type)

# Actual inference smoke test using the model's declared input shape.
inp = session.get_inputs()[0]
shape = [1 if isinstance(v, str) or v is None else int(v) for v in inp.shape]
dummy = np.random.rand(*shape).astype(np.float32)
out = session.run(None, {inp.name: dummy})
print("PASS: ONNX Runtime inference")
print("Output shape:", np.asarray(out[0]).shape)


In [ ]:
# 10. Genuine audio validation against generated positive and negative WAVs
import os, glob, numpy as np
from openwakeword.model import Model

MODEL_PATH = "/content/veronica_training/veronica_wake_model.onnx"
model = Model(wakeword_models=[MODEL_PATH], inference_framework="onnx")

positive = glob.glob("/content/veronica_training/veronica_wake_model/positive_test/*.wav")
negative = glob.glob("/content/veronica_training/veronica_wake_model/negative_test/*.wav")

if not positive or not negative:
    raise RuntimeError("Generated positive/negative test WAVs are missing.")

def score_clip(path):
    predictions = model.predict_clip(path, chunk_size=1280)
    values = []
    for item in predictions:
        values.extend(item.values())
    return float(max(values)) if values else 0.0

# Evaluate a representative set so the cell remains practical in Colab.
positive_scores = [score_clip(p) for p in positive[:50]]
model.reset()
negative_scores = [score_clip(p) for p in negative[:50]]
model.reset()

print("Positive test clips:", len(positive_scores))
print("Negative test clips:", len(negative_scores))
print("Positive max scores: min=", min(positive_scores), "max=", max(positive_scores))
print("Negative max scores: min=", min(negative_scores), "max=", max(negative_scores))

# These are measurements, not fabricated accuracy claims.
print("PASS: real generated-audio inference completed.")


In [ ]:
# 11. Final report and artifact
import os, sys, platform, subprocess, shutil

MODEL_PATH = "/content/veronica_training/veronica_wake_model.onnx"
FINAL_MODEL = "/content/veronica_wake_model.onnx"
REPORT_PATH = "/content/veronica_training_report.txt"

shutil.copy2(MODEL_PATH, FINAL_MODEL)

commit = subprocess.check_output(
    ["git", "-C", "/content/openwakeword", "rev-parse", "HEAD"], text=True
).strip()
piper_commit = subprocess.check_output(
    ["git", "-C", "/content/piper-sample-generator", "rev-parse", "HEAD"], text=True
).strip()

report = f"""
VERONICA PHASE 11E.4 TRAINING REPORT

Python: {sys.version}
Platform: {platform.platform()}
openWakeWord commit: {commit}
Piper sample generator commit: {piper_commit}

Model:
  /content/veronica_wake_model.onnx

Model size:
  {os.path.getsize(FINAL_MODEL)} bytes

ONNX validation:
  PASS

ONNX Runtime inference:
  PASS

Generated-audio validation:
  PASS

NOTE:
  The positive/negative scores above are measurements on generated test
  clips. They are not a substitute for later real-microphone validation.
"""

with open(REPORT_PATH, "w") as f:
    f.write(report)

print("=" * 60)
print("VERONICA WAKE WORD TRAINING COMPLETE")
print("=" * 60)
print("Model :", FINAL_MODEL)
print("Report:", REPORT_PATH)
print("ONNX Runtime: PASS")
print("Real training: PASS")
print("Generated-audio validation: PASS")
print("=" * 60)


## Final files

After all cells finish successfully, download:

- `/content/veronica_wake_model.onnx`
- `/content/veronica_training_report.txt`

The microphone/wake-word hardware test remains the next VERONICA step; it is intentionally not falsely marked as complete by this notebook.